# Tuning and Training

In this notebook it will be train a CNF on the reduced data

In [ ]:
import torch
from time import perf_counter

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))
else:
    device = torch.device("cpu")
    scaler = torch.amp.GradScaler(enabled=(device.type == 'cpu'))

/tmp/ipykernel_3196/673347007.py:9: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = torch.amp.GradScaler(enabled=(device.type == 'cpu'))


In [5]:
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

PyTorch version: 2.12.0+cpu
Device: cpu


## Tuning Hyperparameters

In [ ]:
lr = [1e-3, 5e-4]
num_flows = [12, 16, 24, 32]
hidden_size = [64, 128, 256, 512]
hidden_depth = [1, 2]
wd = [1e-5]

tuning_epochs = 50

In [ ]:
t0_tun = perf_counter()

best_hyperparams = tuning_parameters(
    train_loader=train_loader, 
    val_loader=val_loader,
    lr=lr, 
    num_flows=num_flows, 
    hidden_size=hidden_size, 
    hidden_depth=hidden_depth, 
    weight_decay=wd,
    epochs=tuning_epochs,
    dim_x=dim_x, 
    dim_y=dim_y, 
    device=device
)

t_tuning = perf_counter() - t0_tun

In [ ]:
print(f">>> Tuning\nExecution Time:\t{t_tuning/60:.2f} min")

>>> Tuning
Execution Time:	2.01


## Train the model

In [ ]:
save_dir = "../results/Stokes_temp"
os.makedirs(save_dir, exist_ok=True)
model_save_path = os.path.join(save_dir, "model_best_temp.pt")

EPOCHS = 200
print_frequency = 10
patience = 20

# Hyperparameters
# lr = best_hyperparams['learning_rate'] # 0.001
# num_flows = best_hyperparams['num_flows'] # 24
# hidden_size = best_hyperparams['hidden_size'] # 128
# hidden_depth = best_hyperparams['hidden_depth'] # 2
# wd = best_hyperparams['weight_decacy'] # 1e-05

lr = 0.001
num_flows = 24
hidden_size = 128
hidden_depth = 2
wd = 1e-05

In [ ]:
print(f"Learning Rate: {lr}\nnum_flows: {num_flows}\nhidden_size: {hidden_size}")
print(f"Hidden Depth: {hidden_depth}\nweight_decay: {wd}")

In [ ]:
# Model definition
flow = NormalizingFlow(dim_x, dim_y, num_flows=num_flows, hidden_size=hidden_size, hidden_depth=hidden_depth, device=device).to(device)
print(flow)

In [ ]:
t0_tra = perf_counter()

train_losses, val_losses = full_train(
    epochs=EPOCHS,
    print_frequency=print_frequency,
    model=flow,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=lr,
    weight_decay=wd,
    patience=patience,
    device=device,
    model_save_path=model_save_path
)

t_training = perf_counter() - t0_tra

In [ ]:
print(f">>> Tuning\nExecution Time:\t{t_training/60:.2f} min")

In [ ]:
MODEL_NAME = 'NF_Stokes.pth'
destination_folder = os.path.join("..", "results", "Stokes")
os.makedirs(destination_folder, exist_ok=True)
save_path = os.path.join(destination_folder, MODEL_NAME)

# Load the parameters and settings
flow.load_state_dict(torch.load(model_save_path, weights_only=True))

checkpoint = {
    'model_state_dict': flow.state_dict(),
    'hyperparameters': {
        'lr': lr,
        'num_flows': num_flows,
        'hidden_size': hidden_size,
        'hidden_depth': hidden_depth,
        'wd': wd
    }
}

torch.save(checkpoint, save_path)

# Clean the previous temp file
if os.path.exists(model_save_path):
    os.remove(model_save_path)